In [2]:
!pip install pyspark

  Using cached pyspark-4.1.1-py2.py3-none-any.whl
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]


In [70]:
from pyspark.sql import SparkSession

# 1. Use the Docker Bridge IP for Linux
jdbc_url = "jdbc:postgresql://localhost:5432/qversity"

# 2. Your credentials (ensure they match your Postgres setup)
properties = {
    "user": "postgres",
    "password": "postgres", # Double-check this matches your DB
    "driver": "org.postgresql.Driver"
}

# 3. Define the query (ensure the alias 'as fintech_data' is present)
query = """
(
    select id, 
    data as data_text,
    load_timestamp
    from raw.raw_fintech_data
) as fintech_data
"""

table_name = "raw.raw_fintech_data"

spark = SparkSession.builder \
    .appName("PostgreSQLExample") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

# 4. Load the data using the 'query' variable instead of the full table name
df = spark.read.jdbc(
    url=jdbc_url,
    table=table_name, 
    properties=properties
)

df.printSchema()
df.show(5)

root
 |-- id: integer (nullable = true)
 |-- data: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)

+---+--------------------+--------------------+
| id|                data|      load_timestamp|
+---+--------------------+--------------------+
|  1|{"lat": -18.55140...|2026-05-14 23:01:...|
|  2|{"lat": -4.476765...|2026-05-14 23:01:...|
|  3|{"lat": 8.009853,...|2026-05-14 23:01:...|
|  4|{"lat": -32.71103...|2026-05-14 23:01:...|
|  5|{"lat": -35.21925...|2026-05-14 23:01:...|
+---+--------------------+--------------------+
only showing top 5 rows


In [71]:
first_row=df.first()["data"]

In [72]:
import json
data_dict = json.loads(first_row)
print(len(data_dict.keys()))

24


In [73]:
print(data_dict)

{'lat': -18.551408, 'lon': -69.638686, 'city': 'La Serna', 'email': 'lucas.brito@gmail.com', 'loans': [{'type': 'business', 'status': 'default', 'loan_id': 'LN-C8E808B5CB', 'currency': 'CLP', 'end_date': '2026-10-03', 'principal': 427870339.81, 'start_date': '2021-10-29', 'term_months': 60, 'days_past_due': 151, 'interest_rate': 22.54, 'collateral_type': 'real_estate', 'monthly_payment': 11949054.7, 'outstanding_balance': 126751790.75}, {'type': 'personal', 'status': 'paid_off', 'loan_id': 'LN-0ADBB6C6F1', 'currency': 'CLP', 'end_date': '10-24-2041', 'principal': 251459861.13, 'start_date': '2022-02-06', 'term_months': 240, 'days_past_due': 0, 'interest_rate': 31.77, 'collateral_type': 'none', 'monthly_payment': 6670004.04, 'outstanding_balance': 0.0}], 'gender': 'F', 'status': 'suspended', 'address': "Pasaje Bolivia 637 Interior 460, Nueva Côte d'Ivoire, HGO 60157", 'country': 'CL', 'accounts': [{'status': 'frozen', 'balance': 292109.25, 'currency': 'USD', 'account_id': 'ACC-405300E15

In [74]:
for key, value in data_dict.items():
    print(key,"=>", value)

print("Loans")
for loan in data_dict["loans"]:
    print(loan) 

lat => -18.551408
lon => -69.638686
city => La Serna
email => lucas.brito@gmail.com
loans => [{'type': 'business', 'status': 'default', 'loan_id': 'LN-C8E808B5CB', 'currency': 'CLP', 'end_date': '2026-10-03', 'principal': 427870339.81, 'start_date': '2021-10-29', 'term_months': 60, 'days_past_due': 151, 'interest_rate': 22.54, 'collateral_type': 'real_estate', 'monthly_payment': 11949054.7, 'outstanding_balance': 126751790.75}, {'type': 'personal', 'status': 'paid_off', 'loan_id': 'LN-0ADBB6C6F1', 'currency': 'CLP', 'end_date': '10-24-2041', 'principal': 251459861.13, 'start_date': '2022-02-06', 'term_months': 240, 'days_past_due': 0, 'interest_rate': 31.77, 'collateral_type': 'none', 'monthly_payment': 6670004.04, 'outstanding_balance': 0.0}]
gender => F
status => suspended
address => Pasaje Bolivia 637 Interior 460, Nueva Côte d'Ivoire, HGO 60157
country => CL
accounts => [{'status': 'frozen', 'balance': 292109.25, 'currency': 'USD', 'account_id': 'ACC-405300E15156', 'branch_code': '

In [6]:
for loan in data_dict["loans"]:
    for key, value in loan.items():
        print(key,"=>", value)
    print("-----")


type => business
status => default
loan_id => LN-C8E808B5CB
currency => CLP
end_date => 2026-10-03
principal => 427870339.81
start_date => 2021-10-29
term_months => 60
days_past_due => 151
interest_rate => 22.54
collateral_type => real_estate
monthly_payment => 11949054.7
outstanding_balance => 126751790.75
-----
type => personal
status => paid_off
loan_id => LN-0ADBB6C6F1
currency => CLP
end_date => 10-24-2041
principal => 251459861.13
start_date => 2022-02-06
term_months => 240
days_past_due => 0
interest_rate => 31.77
collateral_type => none
monthly_payment => 6670004.04
outstanding_balance => 0.0
-----


In [7]:
for account in data_dict["accounts"]:
    for key, value in account.items():
        print(key,"=>", value)
    print("-----")

status => frozen
balance => 292109.25
currency => USD
account_id => ACC-405300E15156
branch_code => BR-692
opened_date => 20260111
account_type => investment
credit_limit => None
interest_rate => 2.19
-----
status => active
balance => 396395.98
currency => USD
account_id => ACC-920C0AE87361
branch_code => BR-771
opened_date => 27/02/2026
account_type => credit_card
credit_limit => 23505.19
interest_rate => 18.19
-----


In [8]:
for transaction in data_dict["transactions"]:
    for key, value in transaction.items():
        print(key,"=>", value)
    print("-----")
print(len(data_dict["transactions"]))

date => 2026-04-11
type => Withdrawal
amount => 14039.03
status => pending
channel => pos
category => insurance
currency => USD
merchant => None
account_id => ACC-405300E15156
description => Acciones madrid vio trata llega debía.
transaction_id => TXN-7A7F1852B5E4
-----
date => 05-05-2026
type => fee
amount => $33593.14
status => completed
channel => mobile
category => shopping
currency => USD
merchant => Sodimac
account_id => ACC-405300E15156
description => Primeras señora habla vivir posición oficial tienen todas.
transaction_id => TXN-BBF37C53AEE4
-----
date => 2026-03-16
type => refund
amount => 9307.66
status => completed
channel => pos
category => investment
currency => USD
merchant => Falabella
account_id => ACC-405300E15156
description => Ley fueron evitar poco.
transaction_id => TXN-68E6F9DE3AA7
-----
date => 2026-04-18
type => payment
amount => 9451.52
status => completed
channel => mobile
category => investment
currency => USD
merchant => Spotify
account_id => ACC-405300E151

In [9]:
dataset_portion = df.limit(10)

In [75]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, col
schema = StructType([
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("city", StringType(), True),
    StructField("email", StringType(), True),
    StructField("loans", StringType(), True), # json
    StructField("accounts", StringType(), True), # json
    StructField("transactions", StringType(), True), # json
    StructField("digital_engagement", StringType(), True), # json
    StructField("credit_info", StringType(), True), # json
    StructField("gender", StringType(), True),
    StructField("status", StringType(), True),
    StructField("address", StringType(), True),
    StructField("country", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("kyc_status", StringType(), True),
    StructField("risk_score", DoubleType(), True),
    StructField("customer_id", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("relationship_manager", StringType(), True)
])
# df_parsed = spark.read.json(df.rdd.map(lambda row: row.data), schema=schema)
df_parsed = df.withColumn("data_parsed", from_json(col("data"), schema)).select("data_parsed.*")
df_parsed.printSchema() 

root
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- loans: string (nullable = true)
 |-- accounts: string (nullable = true)
 |-- transactions: string (nullable = true)
 |-- digital_engagement: string (nullable = true)
 |-- credit_info: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- status: string (nullable = true)
 |-- address: string (nullable = true)
 |-- country: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- risk_score: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- relationship_manager: string (nullable =

# Deduplication Strategy 1
Based on the last result, we have 5000 distinct IDs, but 5100 records were ingested, as IDs must be unique, there are 100 duplicate recods.

In [76]:
# selecting only the duplicate rows
duplicate_rows_parsed=df_parsed.groupBy("customer_id").count().filter(col("count") > 1).orderBy(col("count").desc())
duplicate_rows_parsed.show()

+------------+-----+
| customer_id|count|
+------------+-----+
|CUST-0004728|    3|
|CUST-0002417|    2|
|CUST-0004995|    2|
|CUST-0000532|    2|
|CUST-0000597|    2|
|CUST-0001399|    2|
|CUST-0001503|    2|
|CUST-0004362|    2|
|CUST-0003752|    2|
|CUST-0000561|    2|
|CUST-0004909|    2|
|CUST-0003407|    2|
|CUST-0002329|    2|
|CUST-0001686|    2|
|CUST-0004789|    2|
|CUST-0000771|    2|
|CUST-0003211|    2|
|CUST-0000377|    2|
|CUST-0001884|    2|
|CUST-0002703|    2|
+------------+-----+
only showing top 20 rows


# Checking Information is Equal
In order to clean the data we have to consider either to merge the data or drop it completely, firstly we will analyze the duplicate data.

In [77]:
from pyspark.sql import functions as F

columns_to_check = set(df_parsed.columns) - set(["customer_id"])

# This is the "Smart Comparison" logic
# It cleans the data on the fly before checking for uniqueness
agg_exprs = [
    F.countDistinct(F.lower(F.trim(F.col(c).cast("string")))).alias(c) 
    for c in columns_to_check
]

# 1. Group by ID and calculate variation counts
variation_counts = df_parsed.groupBy("customer_id").agg(*agg_exprs)

# 2. Transform the wide table into a long "Report"
stack_string = f"stack({len(columns_to_check)}, " + \
               ", ".join([f"'{c}', {c}" for c in columns_to_check]) + \
               ") as (column_name, unique_count)"

# 3. Filter for 'unique_count > 1' (meaning the data is actually DIFFERENT)
actual_discrepancies = variation_counts.select(
    "customer_id", 
    F.expr(stack_string)
).filter("unique_count > 1")

# Show the results
if actual_discrepancies.count() == 0:
    print("Success! All duplicates contain the same data (ignoring case and spaces).")
else:
    print("Found actual data differences in these IDs/Columns:")
    actual_discrepancies.show(n=100, truncate=False)

Success! All duplicates contain the same data (ignoring case and spaces).


In [78]:
df_parsed = df_parsed.select("*").dropDuplicates(["customer_id"])
df_parsed.count()

5000

In [28]:
for row in df_parsed.take(5):
    print(row)

account_ids = df_parsed.select("customer_id").distinct().collect()
account_ids = [row["customer_id"] for row in account_ids]
print(account_ids)

Row(lat=-11.357, lon=-35.641528, city='Brasilia', email='casandra.villalobos@gmail.com', loans='[{"type":"education","status":"default","loan_id":"LN-1428244D92","currency":"USD","end_date":"2024-07-16","principal":467452.95,"start_date":"2023-07-22","term_months":12,"days_past_due":243,"interest_rate":19.25,"collateral_type":"none","monthly_payment":43134.66,"outstanding_balance":391480.59},{"type":"mortgage","status":"current","loan_id":"LN-7299B81A8C","currency":"BRL","end_date":"2033-10-03","principal":1440199.16,"start_date":"2023-11-25","term_months":120,"days_past_due":0,"interest_rate":10.47,"collateral_type":"none","monthly_payment":19409.14,"outstanding_balance":458237.28}]', accounts='[{"status":"closed","balance":172843.56,"currency":"USD","account_id":"ACC-3F3F63EE6C0B","branch_code":"BR-360","opened_date":"2025-09-11","account_type":"savings","credit_limit":null,"interest_rate":19.77},{"status":"frozen","balance":845.64,"currency":"USD","account_id":"ACC-A7D4F2BC48B7","br

['CUST-0004237', 'CUST-0000519', 'CUST-0000183', 'CUST-0002152', 'CUST-0000198', 'CUST-0002598', 'CUST-0000181', 'CUST-0003727', 'CUST-0003510', 'CUST-0004518', 'CUST-0003973', 'CUST-0001968', 'CUST-0000274', 'CUST-0002686', 'CUST-0000647', 'CUST-0003368', 'CUST-0003511', 'CUST-0002483', 'CUST-0000431', 'CUST-0001176', 'CUST-0001846', 'CUST-0003261', 'CUST-0004506', 'CUST-0004142', 'CUST-0003882', 'CUST-0000434', 'CUST-0004415', 'CUST-0000277', 'CUST-0001534', 'CUST-0002623', 'CUST-0000746', 'CUST-0002903', 'CUST-0003212', 'CUST-0002038', 'CUST-0001404', 'CUST-0002473', 'CUST-0000169', 'CUST-0001934', 'CUST-0004105', 'CUST-0002613', 'CUST-0001616', 'CUST-0002910', 'CUST-0001793', 'CUST-0001220', 'CUST-0002644', 'CUST-0004954', 'CUST-0001161', 'CUST-0002800', 'CUST-0002522', 'CUST-0004151', 'CUST-0004409', 'CUST-0000308', 'CUST-0000587', 'CUST-0000698', 'CUST-0001548', 'CUST-0002417', 'CUST-0004718', 'CUST-0004238', 'CUST-0001117', 'CUST-0002564', 'CUST-0003518', 'CUST-0000843', 'CUST-0

In [79]:
from pyspark.sql.functions import explode, from_json, col
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, DoubleType
df_with_array_loan = df_parsed.withColumn("loans_array", from_json(col("loans"), ArrayType(StructType([
    StructField("loan_id", StringType(), True),
    StructField("type", StringType(), True),
    StructField("status", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("end_date", StringType(), True),
    StructField("principal", DoubleType(), True),
    StructField("start_date", StringType(), True),
    StructField("term_months", DoubleType(), True),
    StructField("days_past_due", DoubleType(), True),
    StructField("interest_rate", DoubleType(), True),
    StructField("collateral_type", StringType(), True),
    StructField("monthly_payment", DoubleType(), True),
    StructField("outstanding_balance", DoubleType(), True)
]))))
df_exploded_loans = df_with_array_loan.withColumn("loan", explode(col("loans_array")))
df_final_loans= df_exploded_loans.select(
    "customer_id",
    "loan.*")

In [80]:
if df_final_loans.select("loan_id").distinct().count() == df_final_loans.count():
    print("All loans have unique IDs!")
else:
    print("Duplicate loan IDs found!")

All loans have unique IDs!


# Loans Information
According to this, all loans have unique IDs across the dataset, which means there are not duplicate rows.

In [81]:
for transaction in data_dict["transactions"]:
    for key, value in transaction.items():
        print(key,"=>", value)
    print("-----")
print(len(data_dict["transactions"]))



df_with_array_transaction = df_parsed.withColumn("transactions_array", from_json(col("transactions"), ArrayType(StructType([
    StructField("transaction_id", StringType(), True),
    StructField("date", StringType(), True),
    StructField("type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("category", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("account_id", StringType(), True),
    StructField("description", StringType(), True),
]))))
df_exploded_transactions = df_with_array_transaction.withColumn("transaction", explode(col("transactions_array")))
df_final_transactions= df_exploded_transactions.select(
    "customer_id",
    "transaction.*")


date => 2026-04-11
type => Withdrawal
amount => 14039.03
status => pending
channel => pos
category => insurance
currency => USD
merchant => None
account_id => ACC-405300E15156
description => Acciones madrid vio trata llega debía.
transaction_id => TXN-7A7F1852B5E4
-----
date => 05-05-2026
type => fee
amount => $33593.14
status => completed
channel => mobile
category => shopping
currency => USD
merchant => Sodimac
account_id => ACC-405300E15156
description => Primeras señora habla vivir posición oficial tienen todas.
transaction_id => TXN-BBF37C53AEE4
-----
date => 2026-03-16
type => refund
amount => 9307.66
status => completed
channel => pos
category => investment
currency => USD
merchant => Falabella
account_id => ACC-405300E15156
description => Ley fueron evitar poco.
transaction_id => TXN-68E6F9DE3AA7
-----
date => 2026-04-18
type => payment
amount => 9451.52
status => completed
channel => mobile
category => investment
currency => USD
merchant => Spotify
account_id => ACC-405300E151

In [85]:
if df_final_transactions.select("transaction_id").distinct().count() == df_final_transactions.count():
    print("All transaction IDs are unique across the dataset.")
else:    print("Duplicate transaction IDs found in the dataset!")

All transaction IDs are unique across the dataset.


In [86]:
for account in data_dict["accounts"]:
    for key, value in account.items():
        print(key,"=>", value)
    print("-----")

df_with_array_account = df_parsed.withColumn("accounts_array", from_json(col("accounts"), ArrayType(StructType([
    StructField("account_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("balance", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("branch_code", StringType(), True),
    StructField("opened_date", StringType(), True),
    StructField("account_type", StringType(), True),
    StructField("credit_limit", DoubleType(), True),
    StructField("interest_rate", DoubleType(), True),
]))))
df_exploded_accounts = df_with_array_account.withColumn("account", explode(col("accounts_array")))
df_final_accounts= df_exploded_accounts.select(
    "customer_id",
    "account.*")

status => frozen
balance => 292109.25
currency => USD
account_id => ACC-405300E15156
branch_code => BR-692
opened_date => 20260111
account_type => investment
credit_limit => None
interest_rate => 2.19
-----
status => active
balance => 396395.98
currency => USD
account_id => ACC-920C0AE87361
branch_code => BR-771
opened_date => 27/02/2026
account_type => credit_card
credit_limit => 23505.19
interest_rate => 18.19
-----


In [55]:
for account in df_final_accounts.take(5):
    print(account)

# df_final_accounts.printSchema()

Row(customer_id='CUST-0000001', account_id='ACC-7B841C599005', status='active', balance=294632.84, currency='USD', branch_code='BR-320', opened_date='2021-01-14', account_type='checking', credit_limit=None, interest_rate=20.33)
Row(customer_id='CUST-0000001', account_id='ACC-385664AAA258', status='closed', balance=171579.81, currency='PEN', branch_code='BR-487', opened_date='2022-01-23', account_type='investment', credit_limit=None, interest_rate=2.87)
Row(customer_id='CUST-0000001', account_id='ACC-94B1E2D62CFF', status='closed', balance=414702.33, currency='USD', branch_code='BR-181', opened_date='03-01-2024', account_type='savings', credit_limit=None, interest_rate=15.65)
Row(customer_id='CUST-0000001', account_id='ACC-DAA30BD045F9', status='cerrado', balance=703233.52, currency='PEN', branch_code='BR-763', opened_date='2021-12-15', account_type='checking', credit_limit=None, interest_rate=11.61)
Row(customer_id='CUST-0000001', account_id='ACC-DE7103A41227', status='closed', balance

In [87]:
if df_final_accounts.select("account_id").distinct().count() == df_final_accounts.count():
    print("All account IDs are unique across the dataset.")
else:    print("Duplicate account IDs found in the dataset!")

All account IDs are unique across the dataset.


In [88]:
columns_to_exclude = ["loans", "accounts", "transactions",]
# columns_to_select = [col for col in df_parsed.columns if col not in columns_to_exclude]
df_final_parsed = df_parsed.drop(*columns_to_exclude)

In [89]:
df_final_parsed.show(5)
df_final_loans.show(5)
df_final_transactions.show(5)
df_final_accounts.show(5)

+----------+-----------+-----------+--------------------+--------------------+--------------------+------+---------+--------------------+-------+----------+----------+----------+----------+------------+-----------+----------------+-------------+----------------+-----------------+--------------------+
|       lat|        lon|       city|               email|  digital_engagement|         credit_info|gender|   status|             address|country| last_name|first_name|kyc_status|risk_score| customer_id|nationality|    phone_number|date_of_birth|customer_segment|registration_date|relationship_manager|
+----------+-----------+-----------+--------------------+--------------------+--------------------+------+---------+--------------------+-------+----------+----------+----------+----------+------------+-----------+----------------+-------------+----------------+-----------------+--------------------+
|   -11.357| -35.641528|   Brasilia|casandra.villalob...|{"last_login_date...|{"currency":"BRL

+------------+-------------+---------+--------+--------+----------+--------------+----------+-----------+-------------+-------------+---------------+---------------+-------------------+
| customer_id|      loan_id|     type|  status|currency|  end_date|     principal|start_date|term_months|days_past_due|interest_rate|collateral_type|monthly_payment|outstanding_balance|
+------------+-------------+---------+--------+--------+----------+--------------+----------+-----------+-------------+-------------+---------------+---------------+-------------------+
|CUST-0000002|LN-1428244D92|education| default|     USD|2024-07-16|     467452.95|2023-07-22|       12.0|        243.0|        19.25|           none|       43134.66|          391480.59|
|CUST-0000002|LN-7299B81A8C| mortgage| current|     BRL|2033-10-03|    1440199.16|2023-11-25|      120.0|          0.0|        10.47|           none|       19409.14|          458237.28|
|CUST-0000004|LN-FC2C60F68B| MORTGAGE| current|     USD|2029-12-13|   

+------------+----------------+----------+-------+--------+--------+-------+-------------+--------+---------+----------------+--------------------+
| customer_id|  transaction_id|      date|   type|  amount|  status|channel|     category|currency| merchant|      account_id|         description|
+------------+----------------+----------+-------+--------+--------+-------+-------------+--------+---------+----------------+--------------------+
|CUST-0000002|TXN-01BCC68DAF12|2025-08-07|deposit|33169.71|  failed| branch|entertainment|     USD|Starbucks|ACC-A7D4F2BC48B7|Esto etapa civil ...|
|CUST-0000002|TXN-5C9DC0068BEF|2026-02-14| refund|15048.19|  failed| branch|entertainment|     USD|PedidosYa|ACC-3F3F63EE6C0B|Papel oposición h...|
|CUST-0000002|TXN-C4746D437753|2026-03-20| refund|33193.03|  failed|    web|          N/A|     USD|Carrefour|ACC-3F3F63EE6C0B|Deleniti modi con...|
|CUST-0000002|TXN-FEC92C6270B1|  20251225|payment|19199.54|reversed| mobile|    insurance|     USD|Carrefour|ACC

+------------+----------------+-------+---------+--------+-----------+-----------+------------+------------+-------------+
| customer_id|      account_id| status|  balance|currency|branch_code|opened_date|account_type|credit_limit|interest_rate|
+------------+----------------+-------+---------+--------+-----------+-----------+------------+------------+-------------+
|CUST-0000001|ACC-7B841C599005| active|294632.84|     USD|     BR-320| 2021-01-14|    checking|        NULL|        20.33|
|CUST-0000001|ACC-385664AAA258| closed|171579.81|     PEN|     BR-487| 2022-01-23|  investment|        NULL|         2.87|
|CUST-0000001|ACC-94B1E2D62CFF| closed|414702.33|     USD|     BR-181| 03-01-2024|     savings|        NULL|        15.65|
|CUST-0000001|ACC-DAA30BD045F9|cerrado|703233.52|     PEN|     BR-763| 2021-12-15|    checking|        NULL|        11.61|
|CUST-0000001|ACC-DE7103A41227| closed|988158.77|     PEN|     BR-801| 2023-03-01|     savings|        NULL|          6.5|
+------------+--

In [90]:
print(df_final_parsed.count())
print(df_final_loans.count())
print(df_final_transactions.count())
print(df_final_accounts.count())

5000


7663


87686


17529


In [95]:
#average transactions per customer
average_transactions = df_final_transactions.groupBy("customer_id").count().agg(F.avg("count").alias("average_transactions_per_customer"))
average_transactions.show()
#average loans per customer
average_loans = df_final_loans.groupBy("customer_id").count().agg(F.avg("count").alias("average_loans_per_customer"))
average_loans.show()

+---------------------------------+
|average_transactions_per_customer|
+---------------------------------+
|                          17.5372|
+---------------------------------+



+--------------------------+
|average_loans_per_customer|
+--------------------------+
|         2.040202342917998|
+--------------------------+



In [99]:
#average accounts per customer
average_accounts = df_final_accounts.groupBy("customer_id").count().agg(F.avg("count").alias("average_accounts_per_customer"))
average_accounts.show()

+-----------------------------+
|average_accounts_per_customer|
+-----------------------------+
|                       3.5058|
+-----------------------------+



In [98]:
from pyspark.sql import functions as F

# 1. Perform the anti-join
# This keeps rows in 'customers' that ARE NOT in 'loans'
customers_without_loans = df_final_parsed.join(
    df_final_loans, 
    on="customer_id", 
    how="left_anti"
)

# 2. Count the results
result_count = customers_without_loans.count()

print(f"Number of customers without loans: {result_count}")

Number of customers without loans: 1244
